<a href="https://colab.research.google.com/github/ParthV303/RAG-Based-PDF-Chatbot/blob/main/chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit
!pip install pyngrok
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pymupdf
!pip install google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 50.1 MB/s eta 0:00:00


In [ ]:
!pip install langchain-text-splitters
import fitz
import faiss
import numpy as np
import streamlit as st
import google.generativeai as genai

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
%%writefile app.py

Writing app.py


In [ ]:
%%writefile app.py

import streamlit as st
import fitz
import faiss
import numpy as np
import google.generativeai as genai

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


API_KEYS = [
    "YOUR_API_KEY",
    "YOUR_API_KEY",
    "YOUR_API_KEY"
]

def generate_with_fallback(prompt):

    for api_key in API_KEYS:

        try:

            genai.configure(
                api_key=api_key
            )

            llm = genai.GenerativeModel(
                "gemini-2.5-flash"
            )

            response = llm.generate_content(
                prompt
            )

            return response.text

        except Exception as e:

            if "429" in str(e) or "ResourceExhausted" in str(e):
                continue

            return f"Error: {e}"

    return "All API keys have reached their quota."


if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

st.title("RAG PDF Chatbot")

pdfs = st.file_uploader(
    "Upload PDFs",
    type="pdf",
    accept_multiple_files=True
)

if pdfs:

    text = ""

    for pdf in pdfs:

        st.write(f"Processing: {pdf.name}")

        text += f"\n\n===== DOCUMENT: {pdf.name} =====\n\n"

        pdf_bytes = pdf.read()

        with open("temp.pdf", "wb") as f:
            f.write(pdf_bytes)

        doc = fitz.open("temp.pdf")

        for page in doc:
            text += page.get_text()

        text += "\n\n"

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_text(text)

    embedding_model = SentenceTransformer(
        "all-MiniLM-L6-v2"
    )

    embeddings = embedding_model.encode(chunks)

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(np.array(embeddings))

    question = st.text_input(
        "Ask Question"
    )

    if question:

        query_embedding = embedding_model.encode(
            [question]
        )

        D, I = index.search(
            np.array(query_embedding),
            k=5
        )

        context = ""

        for idx in I[0]:
            context += chunks[idx] + "\n"

        last_chat = ""

        if len(st.session_state.chat_history) > 0:

            last = st.session_state.chat_history[-1]

            last_chat= f"""
        previous Question:
        {last['question']}

        previous Answer:
        {last['answer']}
        """



        prompt = f"""
Answer ONLY from the context below.

Context:
{context}

{last_chat}

current Question:
{question}

If the current question contains words like:
it, its, they, them, that, those

then use the Previous Question and Previous Answer to understand the reference.

Otherwise answer using only the current question and context.

If the answer is not found in the context,
reply:
Information not found in PDF.
"""

        answer = generate_with_fallback(
            prompt
        )

        st.session_state.chat_history.append(
            {
                "question": question,
                "answer": answer
            }
        )

        st.subheader("Answer")

        st.write(answer)

Overwriting app.py


In [ ]:
!pip install streamlit

In [ ]:
!pip install langchain
!pip install langchain-community

In [ ]:
!pip install langchain-text-splitters

In [ ]:
!pkill streamlit

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!tail -50 /content/logs.txt



2026-06-05 12:47:57.086 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.142.250.55:8501



In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token('3EexzzFHV45cmror8toSW6nddVx_6SdH3Hqp1Ybfquhagin2v')

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://only-treat-safehouse.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://only-treat-safehouse.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!pip install sentence-transformers